# Prepare French analogy inputs

Build `all_cross_instances.parquet` for the French inflection analogy task. Joins `suffix_pairs_top10.csv` (per-row morphological interpretations of (base_orth, derived_orth) pairs) and `suffix_directions_report.csv` (corpus-frequency-supported (suffix_ipa, base_cell, derived_cell) directions) against the MLS French state-space spec and per-token morph tags (`word_morph_detail`, produced by issue #2).

Two join paths:
- **True-friend**: filter directions by frequency → join on `suffix_ipa` → require (`base_cell`, `derived_cell`) consistent with the row's tag sets → require per-token spaCy tag to equal the chosen cell.
- **False-friend**: directions report does not list accidental phonological pairs, so skip it. Take any audio instance of base/derived whose per-token spaCy tag is in the row's `base_tag_set` / `derived_tag_set`. Per-token tag fills the `base_cell` / `derived_cell` columns.

See `notes/analogy-french-design.md` for the full design.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import defaultdict
from pathlib import Path

import datasets
import numpy as np
import pandas as pd

from src.analysis.state_space import StateSpaceAnalysisSpec

In [ ]:
dataset_name = "mls_french-dev"
base_model = "w2v2_pc_fr_8"
state_space_specs_path = f"outputs/state_space_specs/{dataset_name}/{base_model}/state_space_specs.h5"
preprocessed_data_path = f"outputs/preprocessed_data/{dataset_name}"
suffix_pairs_path = "data/french_morphology/suffix_pairs_top10.csv"
suffix_directions_path = "data/french_morphology/suffix_directions_report.csv"
output_dir = f"outputs/analogy/inputs/{dataset_name}/{base_model}"

# Frequency thresholds for filtering the directions report (see design doc).
min_n_stems = 50
min_total_count = 100

seed = 1234

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
np.random.seed(seed)

## Load state-space spec and preprocessed dataset

In [ ]:
state_space_spec = StateSpaceAnalysisSpec.from_hdf5(state_space_specs_path, "word")
label_to_idx = {l: i for i, l in enumerate(state_space_spec.labels)}
print(f"state_space_spec: {len(state_space_spec.labels)} labels, "
      f"{sum(len(s) for s in state_space_spec.target_frame_spans)} instances")

In [ ]:
ds = datasets.Dataset.load_from_disk(preprocessed_data_path)
assert "word_morph_detail" in ds.column_names, (
    f"{preprocessed_data_path} has no word_morph_detail column \u2014 "
    "run issue #2's tag_morphology_mls_french pipeline first.")

In [ ]:
# Phonemic forms per (label, instance_idx) for diagnostics + run notebook
cuts_phones = (
    state_space_spec.cuts.xs("phoneme", level="level")
    .groupby(["label", "instance_idx"]).description.agg(' '.join)
    .rename("phones"))
print(f"cuts_phones: {len(cuts_phones)} (label, instance) entries")

## Build (label, instance_idx) → (item_idx, word_position, csv_cell) map

Walks the preprocessed dataset and replays the iteration order of `compute_word_state_space_no_syllable` so the `instance_idx` derived here matches the spec exactly. An explicit assertion guards the coupling.

In [ ]:
instance_counter: dict[str, int] = defaultdict(int)
instance_info: dict[tuple[str, int], dict] = {}
for i in range(len(ds)):
    row = ds[i]
    for j, w in enumerate(row["word_detail"]["utterance"]):
        if not w:
            continue
        if not row["word_phonemic_detail"][j]:
            continue
        idx = instance_counter[w]
        instance_counter[w] += 1
        wmd = row["word_morph_detail"][j] or {}
        instance_info[(w, idx)] = {
            "item_idx": i,
            "word_position": j,
            "csv_cell": wmd.get("csv_cell"),
            "alignment_confidence": wmd.get("alignment_confidence"),
        }
for label, spans in zip(state_space_spec.labels, state_space_spec.target_frame_spans):
    assert instance_counter[label] == len(spans), (
        f"Instance count mismatch for {label!r}: walked dataset and got "
        f"{instance_counter[label]}, but spec has {len(spans)}. "
        "compute_word_state_space_no_syllable and this iteration must apply "
        "the same skip rules.")
print(f"instance_info: {len(instance_info)} (label, instance) entries (counts match spec)")

In [ ]:
# Index audio instances by (orth, csv_cell) for the true-friend join, and
# by orth alone for the false-friend path.
orth_to_instances_by_cell: dict[tuple[str, str], list[int]] = defaultdict(list)
orth_to_all_instances: dict[str, list[int]] = defaultdict(list)
for (label, idx), info in instance_info.items():
    orth_to_all_instances[label].append(idx)
    if info["csv_cell"] is not None:
        orth_to_instances_by_cell[(label, info["csv_cell"])].append(idx)
print(f"orth_to_instances_by_cell: {len(orth_to_instances_by_cell)} (orth, cell) entries")

## Load CSVs and restrict to the corpus

In [ ]:
suffix_pairs = pd.read_csv(suffix_pairs_path)
directions = pd.read_csv(suffix_directions_path)
print(f"suffix_pairs: {len(suffix_pairs)} rows; directions: {len(directions)} rows")

In [ ]:
# Keep only true_friend / false_friend (drop missing_base) and rows with both orths attested in the corpus.
suffix_pairs = suffix_pairs[suffix_pairs.type.isin(["true_friend", "false_friend"])]
suffix_pairs = suffix_pairs.dropna(subset=["base_orth", "derived_orth"]).copy()
label_set = set(state_space_spec.labels)
suffix_pairs = suffix_pairs[
    suffix_pairs.base_orth.isin(label_set) & suffix_pairs.derived_orth.isin(label_set)
].reset_index(drop=True).copy()
print(f"suffix_pairs after corpus filter: {len(suffix_pairs)} rows "
      f"({suffix_pairs.type.value_counts().to_dict()})")

In [ ]:
def parse_tag_set(s):
    if pd.isna(s) or s == "":
        return frozenset()
    return frozenset(s.split(" ; "))

suffix_pairs["base_tag_set"] = suffix_pairs.base_tags.apply(parse_tag_set)
suffix_pairs["derived_tag_set"] = suffix_pairs.derived_tags.apply(parse_tag_set)

## True-friend path

Filter directions by frequency, join on `suffix_ipa`, require (`base_cell`, `derived_cell`) ∈ row's tag sets, then expand to (base_inst × derived_inst) where per-token spaCy tag matches the chosen cell.

In [ ]:
dirs_keep = directions[
    (directions.n_stems >= min_n_stems)
    & (directions.total_base_count >= min_total_count)
    & (directions.total_derived_count >= min_total_count)
].copy()
print(f"directions kept after freq filter: {len(dirs_keep)}/{len(directions)}")

In [ ]:
tf_pairs = suffix_pairs[suffix_pairs.type == "true_friend"]
tf_joined = tf_pairs.merge(
    dirs_keep[["suffix_ipa", "base_cell", "derived_cell",
               "n_stems", "total_base_count", "total_derived_count"]],
    on="suffix_ipa", how="inner")
tf_mask = tf_joined.apply(
    lambda r: r.base_cell in r.base_tag_set and r.derived_cell in r.derived_tag_set,
    axis=1)
tf_consistent = tf_joined[tf_mask].copy() if len(tf_joined) else tf_joined.copy()
print(f"true-friend (orth-pair, direction) tuples consistent with row tags: {len(tf_consistent)}")

In [ ]:
tf_rows = []
for r in tf_consistent.itertuples(index=False):
    for bi in orth_to_instances_by_cell.get((r.base_orth, r.base_cell), []):
        for di in orth_to_instances_by_cell.get((r.derived_orth, r.derived_cell), []):
            tf_rows.append({
                "suffix_ipa": r.suffix_ipa,
                "suffix_rank": r.suffix_rank,
                "type": r.type,
                "base": r.base_orth,
                "inflected": r.derived_orth,
                "base_cell": r.base_cell,
                "derived_cell": r.derived_cell,
                "base_instance_idx": bi,
                "inflected_instance_idx": di,
                "n_stems": r.n_stems,
                "total_base_count": r.total_base_count,
                "total_derived_count": r.total_derived_count,
                "base_lemma": r.base_lemma,
                "derived_lemma": r.derived_lemma,
                "base_ipa_csv": r.base_ipa,
                "derived_ipa_csv": r.derived_ipa,
            })
print(f"true-friend expanded rows: {len(tf_rows)}")

## False-friend path

No `directions_report` join. For each false_friend orth pair, enumerate base / derived audio instances whose per-token spaCy tag is in the row's `base_tag_set` / `derived_tag_set`. The chosen `base_cell` / `derived_cell` is the per-token tag itself.

On a tiny dev set this typically expands to 0 rows because spaCy's contextual tags rarely match the noun-role tag sets used by the CSV's false-friend interpretations (e.g. "fasse" tagged as subjunctive verb, false-friend row claims masc-noun). With a larger corpus, false-friend rows accumulate from contexts where the surface morphology aligns by chance.

In [ ]:
ff_pairs = suffix_pairs[suffix_pairs.type == "false_friend"]
ff_rows = []
for r in ff_pairs.itertuples(index=False):
    base_candidates = [
        (idx, instance_info[(r.base_orth, idx)]["csv_cell"])
        for idx in orth_to_all_instances.get(r.base_orth, [])
        if instance_info[(r.base_orth, idx)]["csv_cell"] in r.base_tag_set
    ]
    derived_candidates = [
        (idx, instance_info[(r.derived_orth, idx)]["csv_cell"])
        for idx in orth_to_all_instances.get(r.derived_orth, [])
        if instance_info[(r.derived_orth, idx)]["csv_cell"] in r.derived_tag_set
    ]
    for bi, bcell in base_candidates:
        for di, dcell in derived_candidates:
            ff_rows.append({
                "suffix_ipa": r.suffix_ipa,
                "suffix_rank": r.suffix_rank,
                "type": r.type,
                "base": r.base_orth,
                "inflected": r.derived_orth,
                "base_cell": bcell,
                "derived_cell": dcell,
                "base_instance_idx": bi,
                "inflected_instance_idx": di,
                "n_stems": np.nan,
                "total_base_count": np.nan,
                "total_derived_count": np.nan,
                "base_lemma": r.base_lemma,
                "derived_lemma": r.derived_lemma,
                "base_ipa_csv": r.base_ipa,
                "derived_ipa_csv": r.derived_ipa,
            })
print(f"false-friend expanded rows: {len(ff_rows)}")

## Assemble `all_cross_instances`

In [ ]:
all_cross_instances = pd.DataFrame(tf_rows + ff_rows)
if len(all_cross_instances) == 0:
    raise SystemExit(
        "No surviving (base, inflected) pairs \u2014 check inputs (likely the corpus "
        "is too small for any CSV orth-pair to be attested with matching tags).")

In [ ]:
all_cross_instances["inflection"] = (
    all_cross_instances.suffix_ipa
    + "::"
    + all_cross_instances.base_cell
    + "\u2192"
    + all_cross_instances.derived_cell)
all_cross_instances["base_idx"] = all_cross_instances.base.map(label_to_idx)
all_cross_instances["inflected_idx"] = all_cross_instances.inflected.map(label_to_idx)

In [ ]:
# Attach corpus phonemic forms (analogous to English notebook's base_phones / inflected_phones)
all_cross_instances = all_cross_instances.merge(
    cuts_phones.rename("base_phones").reset_index().rename(
        columns={"label": "base", "instance_idx": "base_instance_idx"}),
    on=["base", "base_instance_idx"], how="left")
all_cross_instances = all_cross_instances.merge(
    cuts_phones.rename("inflected_phones").reset_index().rename(
        columns={"label": "inflected", "instance_idx": "inflected_instance_idx"}),
    on=["inflected", "inflected_instance_idx"], how="left")

In [ ]:
# false_friends are excluded from the headline accuracy summary (cf. English notebook's exclude_main convention).
all_cross_instances["exclude_main"] = all_cross_instances.type == "false_friend"

preferred = ["inflection", "suffix_ipa", "type", "base_cell", "derived_cell",
             "base", "inflected", "base_idx", "inflected_idx",
             "base_instance_idx", "inflected_instance_idx",
             "base_phones", "inflected_phones",
             "base_ipa_csv", "derived_ipa_csv",
             "base_lemma", "derived_lemma",
             "n_stems", "total_base_count", "total_derived_count",
             "suffix_rank", "exclude_main"]
all_cross_instances = all_cross_instances[preferred]
print(f"Final all_cross_instances: {len(all_cross_instances)} rows")
print(all_cross_instances.groupby(["type", "suffix_ipa"]).size())

In [ ]:
all_cross_instances.head(15)

## Type-level summary (inflection_results)

In [ ]:
inflection_results_df = (
    all_cross_instances
    .groupby(["inflection", "suffix_ipa", "type", "base_cell", "derived_cell",
              "base", "inflected"], dropna=False)
    .size().rename("n_instance_pairs").reset_index())
print(f"inflection_results: {len(inflection_results_df)} type-level rows")
inflection_results_df

## Save

In [ ]:
all_cross_instances.to_parquet(f"{output_dir}/all_cross_instances.parquet")
inflection_results_df.to_parquet(f"{output_dir}/inflection_results.parquet")
state_space_spec.to_hdf5(f"{output_dir}/state_space_spec.h5")
print(f"Saved to {output_dir}/")

## Diagnostics

Two diagnostics for sanity-checking the pipeline before scaling to a larger corpus:
1. **Filter decomposition** — rows in / rows out at each stage, separately for true-friend and false-friend paths. Shows which stage is doing the heavy lifting.
2. **False-friend spot-check** — for each dropped FF orth pair, print the CSV's tag-set expectation, what spaCy actually returned per audio token, and the surrounding utterance context. Lets you verify whether spaCy is right (homophone resolved by context, FF rejection appropriate) or wrong (FF detection degraded). At dev scale these are smoke-test outputs; re-inspect at train scale.

In [ ]:
# Filter decomposition
sp_raw = pd.read_csv(suffix_pairs_path)
n0 = len(sp_raw)
n1 = sp_raw.type.isin(['true_friend','false_friend']).sum()
sp_step1 = sp_raw[sp_raw.type.isin(['true_friend','false_friend'])].dropna(subset=['base_orth','derived_orth'])
n2 = len(sp_step1)
sp_step2 = sp_step1[sp_step1.base_orth.isin(label_set) & sp_step1.derived_orth.isin(label_set)]
n3 = len(sp_step2)

print('FILTER DECOMPOSITION')
print('=' * 60)
print(f'  suffix_pairs (raw)                            : {n0:>9}')
print(f'  after type ∈ {{true_friend, false_friend}}        : {n1:>9}  (-{n0-n1}, missing_base dropped)')
print(f'  after dropna(base_orth, derived_orth)         : {n2:>9}  (-{n1-n2})')
print(f'  after orth ∈ corpus                          : {n3:>9}  (-{n2-n3})')
print(f'    ├─ true_friend : {(sp_step2.type=="true_friend").sum()}')
print(f'    └─ false_friend: {(sp_step2.type=="false_friend").sum()}')

print('\n--- TRUE-FRIEND PATH ---')
n_dirs = len(directions)
n_dirs_keep = len(dirs_keep)
print(f'  directions (raw)                              : {n_dirs:>9}')
print(f'  directions after freq filter                  : {n_dirs_keep:>9}  (-{n_dirs-n_dirs_keep})')
print(f'  TF ⋈ dirs_keep on suffix_ipa                   : {len(tf_joined):>9}')
print(f'  after tag-consistency                         : {len(tf_consistent):>9}  (-{len(tf_joined)-len(tf_consistent)})')
print(f'  after per-token spaCy tag match (× instances)  : {len(tf_rows):>9}')

print('\n--- FALSE-FRIEND PATH ---')
print(f'  false_friend rows after corpus filter         : {(sp_step2.type=="false_friend").sum():>9}')
print(f'  after per-token spaCy tag ∈ row tag_sets       : {len(ff_rows):>9}')

In [ ]:
# False-friend spot-check: for each dropped FF orth pair, show CSV expectation vs. spaCy reality
ff_pairs_raw = sp_step2[sp_step2.type == 'false_friend'].copy()
ff_pairs_raw['base_tag_set'] = ff_pairs_raw.base_tags.apply(parse_tag_set)
ff_pairs_raw['derived_tag_set'] = ff_pairs_raw.derived_tags.apply(parse_tag_set)
ff_distinct = ff_pairs_raw.drop_duplicates(['base_orth','derived_orth'])

print(f'False-friend orth pairs (corpus-attested): {len(ff_distinct)}\n')
for r in ff_distinct.itertuples(index=False):
    all_rows = ff_pairs_raw[(ff_pairs_raw.base_orth==r.base_orth) & (ff_pairs_raw.derived_orth==r.derived_orth)]
    base_union = frozenset().union(*all_rows.base_tag_set)
    derived_union = frozenset().union(*all_rows.derived_tag_set)
    print(f'=== ({r.base_orth!r}, {r.derived_orth!r}) suffix_ipa={r.suffix_ipa!r} ===')
    print(f'  CSV base_tag_set    (union): {sorted(base_union)}')
    print(f'  CSV derived_tag_set (union): {sorted(derived_union)}')
    for side, w, expected in [('base', r.base_orth, base_union), ('derived', r.derived_orth, derived_union)]:
        for idx in orth_to_all_instances.get(w, []):
            info = instance_info[(w, idx)]
            item = ds[info['item_idx']]
            wp = info['word_position']
            utt_words = item['word_detail']['utterance']
            lo, hi = max(0, wp-3), min(len(utt_words), wp+4)
            ctx = ' '.join(['**'+x+'**' if k==wp else x for k,x in enumerate(utt_words[lo:hi], start=lo)])
            mark = '✓' if info['csv_cell'] in expected else '✗'
            print(f"  {side} inst {idx}: spacy={info['csv_cell']!r} {mark}  ctx: ...{ctx}...")
    print()

### FF=0 finding (dev set)

On `mls_french-dev` the FF spot-check finds **spaCy consistently right**: for every dropped FF orth pair, the audio context confirms the spaCy tag (e.g. *fasse* in "que je le **fasse** mourir" is unambiguously subjunctive verb; *t* in "cria **t** elle" is a euphonic clitic letter that spaCy correctly refuses to UPOS-tag; *dit* and *fait* are clearly verbs in their contexts).

The 0-row outcome is therefore **principled**, not a probe degradation. But this also means **FF rows may stay near-zero at train scale**, because the CSV's FF rows describe alternative morphological readings (typically rare noun-of-letter or fragmentary readings) that rarely surface in narrated text. If the FF experiment turns out to be empty at scale too, the design needs revisiting — either accept that the FF probe is sparse, or relax to per-orth-only (ignore CSV's tag interpretation, just take any base/derived pairing). Flagging for follow-up; do not run on train split until we've eyeballed the FF spot-check on a larger dev set.